# Sentiment Analysis using Bidirectional LSTM with Word2Vec

## CSE 4221 — Natural Language Processing Assignment

**Model:** Bidirectional LSTM  
**Embeddings:** Word2Vec (300-dimensional, trained on corpus)  
**Dataset:** IMDB Movie Review Dataset  
**Task:** Binary Sentiment Classification (Positive / Negative)

---
## 1. Import Libraries

In [3]:
!pip install gensim
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

from gensim.models import Word2Vec

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt
print(f'TensorFlow version: {tf.__version__}')
print('All libraries imported successfully.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 18.7 MB/s eta 0:00:00:00:0100:01
TensorFlow version: 2.19.0
All libraries imported successfully.


---
## 2. Load the IMDB Dataset

In [4]:
# Download dataset from Google Drive
import gdown

url = 'https://drive.google.com/uc?id=1GOOxQZSmtQZ4q-UPVh87DHOgI79cx9vI'
output = 'IMDB-Dataset.csv'
gdown.download(url, output, quiet=False)

# Load dataset
df = pd.read_csv('IMDB-Dataset.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nSentiment Distribution:\n{df["sentiment"].value_counts()}')

# Remove duplicates
duplicates = df.duplicated().sum()
if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Removed {duplicates} duplicates. New shape: {df.shape}')

df.head()

Downloading...
From: https://drive.google.com/uc?id=1GOOxQZSmtQZ4q-UPVh87DHOgI79cx9vI
To: /content/IMDB-Dataset.csv
100%|██████████| 66.2M/66.2M [00:00<00:00, 76.8MB/s]


Dataset shape: (50000, 2)
Columns: ['review', 'sentiment']

Sentiment Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
Removed 418 duplicates. New shape: (49582, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


---
## 3. Text Preprocessing

In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Clean and preprocess a single review."""
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = text.lower()
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

print('Preprocessing reviews...')
df['cleaned_review'] = df['review'].apply(preprocess_text)
print('Preprocessing complete.')

# Encode labels
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print(f'\nSample cleaned review:\n{df["cleaned_review"].iloc[0][:150]}')

Preprocessing reviews...
Preprocessing complete.

Sample cleaned review:
one reviewer mentioned watching episode hooked right exactly happened first thing struck brutality unflinching scene violence set right word trust sho


---
## 4. Train-Test Split (80:20)

In [6]:
X = df['cleaned_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {len(X_train)}')
print(f'Test set:     {len(X_test)}')

Training set: 39665
Test set:     9917


---
## 5. Tokenization and Sequence Padding

We tokenize the text and convert each review into a fixed-length sequence of integer indices.

In [7]:
# Parameters
VOCAB_SIZE = 20000
MAX_LEN = 200    # Maximum sequence length
EMBED_DIM = 300  # Word2Vec embedding dimension

# Tokenize
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

word_index = tokenizer.word_index
print(f'Vocabulary size: {len(word_index)}')
print(f'Training sequences shape: {X_train_pad.shape}')
print(f'Test sequences shape:     {X_test_pad.shape}')

Vocabulary size: 82956
Training sequences shape: (39665, 200)
Test sequences shape:     (9917, 200)


---
## 6. Train Word2Vec Embeddings

We train a **Word2Vec** model on the training corpus to learn 300-dimensional word embeddings that capture semantic relationships between words.

In [8]:
# Prepare tokenized sentences for Word2Vec
train_sentences = [text.split() for text in X_train]

# Train Word2Vec model
w2v_model = Word2Vec(
    sentences=train_sentences,
    vector_size=EMBED_DIM,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    sg=0  # CBOW (Continuous Bag of Words)
)

print(f'Word2Vec vocabulary size: {len(w2v_model.wv)}')
print(f'Embedding dimension: {w2v_model.wv.vector_size}')

Word2Vec vocabulary size: 50028
Embedding dimension: 300


In [9]:
# Build embedding matrix
vocab_size = min(VOCAB_SIZE, len(word_index) + 1)
embedding_matrix = np.zeros((vocab_size, EMBED_DIM))

found = 0
not_found = 0
for word, idx in word_index.items():
    if idx >= vocab_size:
        continue
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]
        found += 1
    else:
        not_found += 1

print(f'Embedding matrix shape: {embedding_matrix.shape}')
print(f'Words found in Word2Vec: {found}')
print(f'Words NOT found (zero vectors): {not_found}')
print(f'Coverage: {found/(found+not_found)*100:.1f}%')

Embedding matrix shape: (20000, 300)
Words found in Word2Vec: 19998
Words NOT found (zero vectors): 1
Coverage: 100.0%


---
## 7. Build Bidirectional LSTM Model

The Bi-LSTM processes the input sequence in both forward and backward directions, capturing context from both sides of each word.

In [10]:
model = Sequential([
    # Pre-trained Word2Vec embeddings (non-trainable)
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=False
    ),
    SpatialDropout1D(0.3),
    
    # Bidirectional LSTM layers
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    
    # Dense classification layers
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary output
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     6,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,000,000 (22.89 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 6,000,000 (22.89 MB)

---
## 8. Train the Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss', patience=3,
    restore_best_weights=True, verbose=1
)

y_train_arr = y_train.values
y_test_arr = y_test.values

history = model.fit(
    X_train_pad, y_train_arr,
    epochs=10, batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop], verbose=1
)

Epoch 1/10
386/558 ━━━━━━━━━━━━━━━━━━━━ 11:58 4s/step - accuracy: 0.7449 - loss: 0.5102

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Prediction and Evaluation

In [ ]:
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

accuracy  = accuracy_score(y_test_arr, y_pred)
precision = precision_score(y_test_arr, y_pred)
recall    = recall_score(y_test_arr, y_pred)
f1        = f1_score(y_test_arr, y_pred)

print('=' * 55)
print('  Bi-LSTM + Word2Vec — RESULTS')
print('=' * 55)
print(f'  Accuracy:  {accuracy:.4f}')
print(f'  Precision: {precision:.4f}')
print(f'  Recall:    {recall:.4f}')
print(f'  F1-Score:  {f1:.4f}')
print('=' * 55)

In [ ]:
print('\nClassification Report:')
print(classification_report(y_test_arr, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test_arr, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
disp.plot(cmap='Greens', ax=ax, values_format='d')
ax.set_title('Confusion Matrix — Bi-LSTM + Word2Vec', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Analysis and Discussion

### Observations
- **Word2Vec embeddings** capture semantic relationships, providing dense 300-dim representations.
- **Bidirectional LSTM** processes sequences in both directions, capturing context from past and future.
- The stacked Bi-LSTM (128 + 64 units) learns hierarchical text representations.

### Advantages
- Dense embeddings encode semantic similarity (e.g., "good" ≈ "great").
- Bi-LSTM captures word order and long-range dependencies.
- The model learns complex non-linear patterns.

### Limitations
- Word2Vec trained only on IMDB corpus limits vocabulary coverage.
- LSTM training is computationally expensive.
- Many hyperparameters to tune (seq length, units, dropout, etc.).